In [0]:
from pyspark.sql.functions import col, to_date

# Carregando as tabelas da Silver e as Dimensões da Gold
df_eventos = spark.table("silver_eventos")
df_orgao = spark.table("gold_dim_orgao")

# Chave de data para a dim_data
df_eventos_prep = df_eventos.withColumn("id_data", to_date(col("dataHoraInicio")))

# JOIN id_orgao_organizador com o id_orgao (da dimensão)
fato_eventos = df_eventos_prep.join(
    df_orgao, 
    df_eventos_prep.id_orgao_organizador == df_orgao.id_orgao,  
    "left"
).select(
    col("id").alias("id_evento"),     
    col("id_orgao"),                  
    col("id_data"),                   
    "descricao",
    "descricaoTipo",
    "situacao",
    "dataHoraInicio",
    "dataHoraFim"
)

# Salvando na Gold
fato_eventos.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_fato_eventos")

display(spark.table("gold_fato_eventos"))